In [37]:
import pandas as pd
import numpy as np
import gc
import io
import os
import csv
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor, as_completed
from multiprocessing import Pool, cpu_count
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

pd.reset_option('display.float_format')
pd.set_option('display.max_colwidth', None)

from config import ROOT, prev_num_aggregations  # lib này được khởi tạo ban đầu dự án

import helpers.view as view
import helpers.EDA as EDA
import modules.utils as utils
import modules.encode as encode

from helpers.cache_clear import cache_clear

get_pickle = utils.get_pickle
get_pickles = utils.get_pickles

In [70]:
used_f0_f101 = pd.read_csv(ROOT + "/.log/_used/5_lightgbm_4_used_2_f0_f101.csv")
used_f0_f102 = pd.read_csv(ROOT + "/.log/_used/5_lightgbm_4_used_2_f0_f102.csv")
used_f0_f103 = pd.read_csv(ROOT + "/.log/_used/5_lightgbm_4_used_2_f0_f103.csv")
used_f0_f104 = pd.read_csv(ROOT + "/.log/_used/5_lightgbm_4_used_2_f0_f104.csv")
used_f0_f105_ = pd.read_csv(ROOT + "/.log/_used/5_lightgbm_4_used_2_f0_f105_f106_f107_f108.csv")
used_f0_f101["data"] = 1
used_f0_f102["data"] = 2
used_f0_f103["data"] = 3
used_f0_f104["data"] = 4
used_f0_f105_["data"] = 5

low_corr_f0_f101 = pd.read_csv(ROOT + "/.log/_used/6_lightgbm_2_low_corr_target_f0_f101.csv")
low_corr_f0_f102 = pd.read_csv(ROOT + "/.log/_used/6_lightgbm_2_low_corr_target_f0_f102.csv")
low_corr_f0_f103 = pd.read_csv(ROOT + "/.log/_used/6_lightgbm_2_low_corr_target_f0_f103.csv")
low_corr_f0_f104 = pd.read_csv(ROOT + "/.log/_used/6_lightgbm_2_low_corr_target_f0_f104.csv")
low_corr_f0_f105_ = pd.read_csv(ROOT + "/.log/_used/6_lightgbm_2_low_corr_target_f0_f105_f106_f107_f108.csv")
low_corr_f0_f101["data"] = 1
low_corr_f0_f102["data"] = 2
low_corr_f0_f103["data"] = 3
low_corr_f0_f104["data"] = 4
low_corr_f0_f105_["data"] = 5

used = pd.concat([used_f0_f101, used_f0_f102, used_f0_f103, used_f0_f104, used_f0_f105_])
low_corr = pd.concat([low_corr_f0_f101, low_corr_f0_f102, low_corr_f0_f103, low_corr_f0_f104, low_corr_f0_f105_])

In [71]:
used.reset_index(drop=True, inplace=True)
low_corr.reset_index(drop=True, inplace=True)

In [72]:
used.sort_values(["data", "chunk", "importance_gain", "importance_split"], ascending=[True, True, False, False], inplace=True)
low_corr.sort_values(["data", "chunk", "importance_gain", "importance_split"], ascending=[True, True, False, False], inplace=True)

In [73]:
used

,chunk,feature,importance_gain,importance_split,data
52,1,f001_EXT_SOURCES_mean,143841.527904,895,1
94,1,f002_ORGANIZATION_TYPE,55839.384460,3552,1
50,1,f001_EXT_SOURCES_sum,29881.451856,492,1
51,1,f001_EXT_SOURCE_3,25148.716304,839,1
10,1,f001_AMT_CREDIT-d-AMT_ANNUITY,20280.165388,1077,1
...,...,...,...,...,...
21774,2,f108_past_payment_67m,0.000000,0,5
21775,2,f108_past_payment_71m,0.000000,0,5
21776,2,f108_past_payment_66m,0.000000,0,5
21777,2,f108_past_payment_63m,0.000000,0,5


In [74]:
low_corr

,chunk,feature,importance_gain,importance_split,data
3,1,f001_AMT_ANNUITY-d-AMT_CREDIT,73799.700112,2597,1
137,1,f002_OCCUPATION_TYPE,33317.495951,1345,1
29,1,f001_DAYS_EMPLOYED-s-DAYS_BIRTH,16526.310772,1148,1
1,1,f001_AMT_ANNUITY,11995.033043,812,1
33,1,f001_DAYS_ID_PUBLISH-d-DAYS_BIRTH,11813.024924,777,1
...,...,...,...,...,...
34990,2,f108_past_payment_52m,0.000000,0,5
34991,2,f108_past_payment_54m,0.000000,0,5
34992,2,f108_past_payment_50m,0.000000,0,5
34993,2,f108_past_payment_49m,0.000000,0,5


## Vì 109_lightgbm_selection lấy importance theo chunk nên việc đánh giá importance giữa các chunk khác nhau có thể không khách quan (có thể nó kém quan trọng so với features khác cùng chunk nhưng tổng thể information gain lại lớn)

In [20]:
def filter_by_threshold(chunk_data):
    threshold = min(np.percentile(chunk_data["importance_gain"], 80), 1000)
    return chunk_data[chunk_data["importance_gain"] > threshold], chunk_data[chunk_data["importance_gain"] <= threshold] # lấy threshold là phân vị 80%, threshold nhỏ nhất là 1000

def split_by_group(df):
    kept_list = []
    dropped_list = []
    for _, chunk in df.groupby(["data", 'chunk']):
        kept, dropped = filter_by_threshold(chunk)
        kept_list.append(kept)
        dropped_list.append(dropped)
    return pd.concat(kept_list), pd.concat(dropped_list)

filtered_used, _filtered_used = split_by_group(used) # gain cao và gain thấp trong used
filtered_low_corr, _filtered_low_corr = split_by_group(used) # gain cao và gain thấp trong low_corr

In [21]:
import re
def read_feather_with_head(file_path):
    return pd.read_feather(file_path).head(HEAD)

def read(filename):
    with open(filename, 'r') as f:
        features = [ROOT + "/data/feature/train/" + line.strip() + ".f" for line in f]
        return features
    
def sanitize_feature_name(name):
    return re.sub(r"[+(),. ]", "_", name)

In [ ]:
# def convert_file_name(path):
#         dir_path, filename = os.path.split(path)
#         name, ext = os.path.splitext(filename)

#         sanitized_name = sanitize_feature_name(name)
#         if sanitized_name.endswith("_f"):
#                 sanitized_name = sanitized_name[:-2]
#         sanitized = sanitized_name + ext

#         new_path = os.path.join(dir_path, sanitized)
#         os.rename(path, new_path)
#         print(f"Renamed: {filename} → {sanitized}")
#         return new_path


# n_thread=12

# prefixes = ["f001", "f101", "f102", "f103", "f104", "f105", "f106", "f107", "f108"]
# feature_paths = []
# paths_dir = "D:/Data Science/data/feature/train"
# for prefix in prefixes:
#         feature_paths += sorted([os.path.join(paths_dir, f) for f in os.listdir(paths_dir) if f.startswith(prefix)])
# # feature_paths = utils.get_feature_paths(prefixes=prefixes)
# chunk_size = n_thread * 50 # 40-50 là vừa đủ load 70 - 80% RAM, 80% CPU và GPU nhận được lượng data phù hợp tránh nghẽn cổ chai
# chunks = [feature_paths[i:i + chunk_size] for i in range(0, len(feature_paths), chunk_size)]

# with ThreadPoolExecutor(max_workers=n_thread) as executor:
#     for file_paths in chunks:
#         futures = [executor.submit(convert_file_name, file_path) for file_path in file_paths]

In [47]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
HEAD=180000
SEED=71
f001 = [
    'f001_NAME_CONTRACT_TYPE',
    'f001_CODE_GENDER',
    'f001_FLAG_OWN_CAR',
    'f001_FLAG_OWN_REALTY',
    'f001_NAME_TYPE_SUITE',
    'f001_NAME_INCOME_TYPE',
    'f001_NAME_EDUCATION_TYPE',
    'f001_NAME_FAMILY_STATUS',
    'f001_NAME_HOUSING_TYPE',
    'f001_OCCUPATION_TYPE',
    'f001_WEEKDAY_APPR_PROCESS_START',
    'f001_ORGANIZATION_TYPE',
    'f001_FONDKAPREMONT_MODE',
    'f001_HOUSETYPE_MODE',
    'f001_WALLSMATERIAL_MODE',
    'f001_EMERGENCYSTATE_MODE',
]

f002 = [
    'f002_NAME_CONTRACT_TYPE',
    'f002_CODE_GENDER',
    'f002_FLAG_OWN_CAR',
    'f002_FLAG_OWN_REALTY',
    'f002_NAME_TYPE_SUITE',
    'f002_NAME_INCOME_TYPE',
    'f002_NAME_EDUCATION_TYPE',
    'f002_NAME_FAMILY_STATUS',
    'f002_NAME_HOUSING_TYPE',
    'f002_OCCUPATION_TYPE',
    'f002_WEEKDAY_APPR_PROCESS_START',
    'f002_ORGANIZATION_TYPE',
    'f002_FONDKAPREMONT_MODE',
    'f002_HOUSETYPE_MODE',
    'f002_WALLSMATERIAL_MODE',
    'f002_EMERGENCYSTATE_MODE',
]

f003 = [
    'f002_NAME_CONTRACT_TYPE',
    'f002_CODE_GENDER',
    'f002_FLAG_OWN_CAR',
    'f002_FLAG_OWN_REALTY',
    'f002_NAME_TYPE_SUITE',
    'f002_NAME_INCOME_TYPE',
    'f002_NAME_EDUCATION_TYPE',
    'f002_NAME_FAMILY_STATUS',
    'f002_NAME_HOUSING_TYPE',
    'f002_OCCUPATION_TYPE',
    'f002_WEEKDAY_APPR_PROCESS_START',
    'f002_ORGANIZATION_TYPE',
    'f002_FONDKAPREMONT_MODE',
    'f002_HOUSETYPE_MODE',
    'f002_WALLSMATERIAL_MODE',
    'f002_EMERGENCYSTATE_MODE',
]


ALL_CAT = f001 + f002 + f003

### tunning nhẹ param vì auc_train và auc_valid đang có dấu hiệu overfit

In [67]:
# param tunning:
param = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.01,
    'max_depth': 6, # or 10 - 15
    'num_leaves': 31, # 63 or 20 - 50
    'max_bin': 255,
    'min_child_weight': 10, # 5 or 10
    'min_data_in_leaf': 150, # 100 or 150 
    'reg_lambda': 1, #0.5 or 0.01 or 0.1 # L2 regularization term on weights.
    'reg_alpha': 0.5, # 0.5  # L1 regularization term on weights.
    'colsample_bytree': 0.7,
    'subsample': 0.6, # 0.5
    # 'nthread': 12,
    'bagging_freq': 1,
    'verbose': 0,
    'seed': SEED,
    # thêm cấu hình cho GPU
    'device_type': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
}

In [60]:
n_thread=12 # cpu 6 cores 12 threads

feature_paths = [ROOT + "/data/feature/train/" + sanitize_feature_name(feature).strip() + ".f" for feature in _filtered_used["feature"].tolist()] # thử importance của _filtered_used gain thấp


feature_paths_size = len(feature_paths)
chunk_size = n_thread * 50 # 40-50 là vừa đủ load 70 - 80% RAM, 80% CPU và GPU nhận được lượng data đều đặn tránh nghẽn cổ chai
chunks = [feature_paths[i:i + chunk_size] for i in range(0, len(feature_paths), chunk_size)]

print(feature_paths[:10])

target = pd.read_feather(utils.get_TARGET_path()).head(HEAD)
target.columns = ["TARGET"]

# multi thread
with ThreadPoolExecutor(max_workers=n_thread) as executor, \
    open(os.path.join(ROOT, f".log/_used/7_lightgbm_filtered_low_imp_used.csv"), "w", newline='') as f_selected:
    
    chunk=1
    writer = csv.writer(f_selected)
    writer.writerow(['chunk', 'feature', 'importance_gain', 'importance_split', 'auc_train', 'auc_valid'])
    n = 0
    
    for file_paths in chunks:
        futures = [executor.submit(read_feather_with_head, file_path) for file_path in file_paths]
        chunk_dfs = [future.result() for future in as_completed(futures)]
        
        X = pd.concat(chunk_dfs, axis=1)
        X.columns = [sanitize_feature_name(name) for name in X.columns]
        
        X_train, X_val, y_train, y_val = train_test_split(
            X, target, test_size=0.2, random_state=SEED
        )

        dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=list(set(X_train.columns) & set(ALL_CAT)))
        dval = lgb.Dataset(X_val, label=y_val, categorical_feature=list(set(X_val.columns) & set(ALL_CAT)))

        model = lgb.train(param, dtrain, valid_sets=[dtrain, dval], valid_names=["train", "val"], num_boost_round=1000, callbacks=[lgb.log_evaluation(100), lgb.early_stopping(100)])
        importance_gain = model.feature_importance(importance_type='gain')
        importance_split = model.feature_importance(importance_type='split')

        feature_names = X.columns
        
        y_pred_train = model.predict(X_train)
        y_pred_val = model.predict(X_val)
        auc_train = roc_auc_score(y_train, y_pred_train)
        auc_val = roc_auc_score(y_val, y_pred_val)

        csv_rows = []
        for i, feature_name in enumerate(feature_names):
            csv_rows.append([chunk, feature_name, importance_gain[i], importance_split[i], auc_train, auc_val])
        
        writer.writerows(csv_rows)
        chunk+=1
        
        n += len(file_paths)
        print(n, " / ", feature_paths_size)

['d:\\Data Science/data/feature/train/f101_approved_AMT_ANNUITY_s_app_AMT_CREDIT_diff_max.f', 'd:\\Data Science/data/feature/train/f101_active_cnt_unpaid_max.f', 'd:\\Data Science/data/feature/train/f101_active_DAYS_LAST_DUE_1ST_VERSION_d_app_DAYS_EMPLOYED_max.f', 'd:\\Data Science/data/feature/train/f002_NAME_EDUCATION_TYPE.f', 'd:\\Data Science/data/feature/train/f101_active_DAYS_FIRST_DUE_d_app_DAYS_BIRTH_min.f', 'd:\\Data Science/data/feature/train/f101_active_DAYS_LAST_DUE_1ST_VERSION_s_app_DAYS_LAST_PHONE_CHANGE_mean.f', 'd:\\Data Science/data/feature/train/f101_active_DAYS_TERMINATION_s_DAYS_LAST_DUE_1ST_VERSION_min.f', 'd:\\Data Science/data/feature/train/f101_active_AMT_CREDIT_d_total_debt_pctchange_mean.f', 'd:\\Data Science/data/feature/train/f101_active_AMT_CREDIT_s_app_AMT_INCOME_TOTAL_max.f', 'd:\\Data Science/data/feature/train/f101_active_DAYS_DECISION_d_app_DAYS_BIRTH_min.f']
Training until validation scores don't improve for 100 rounds
[100]	train's auc: 0.709961	val'

In [62]:
n_thread=12 # cpu 6 cores 12 threads

feature_paths = [ROOT + "/data/feature/train/" + sanitize_feature_name(feature).strip() + ".f" for feature in filtered_used["feature"].tolist()] # thử importance của filtered_used gain cao


feature_paths_size = len(feature_paths)
chunk_size = n_thread * 50 # 40-50 là vừa đủ load 70 - 80% RAM, 80% CPU và GPU nhận được lượng data phù hợp tránh nghẽn cổ chai
chunks = [feature_paths[i:i + chunk_size] for i in range(0, len(feature_paths), chunk_size)]

target = pd.read_feather(utils.get_TARGET_path()).head(HEAD)
target.columns = ["TARGET"]

# multi thread
with ThreadPoolExecutor(max_workers=n_thread) as executor, \
    open(os.path.join(ROOT, f".log/_used/7_lightgbm_filtered_high_imp_used.csv"), "w", newline='') as f_selected:
    
    chunk=1
    writer = csv.writer(f_selected)
    writer.writerow(['chunk', 'feature', 'importance_gain', 'importance_split', 'auc_train', 'auc_valid'])
    n = 0
    
    for file_paths in chunks:
        futures = [executor.submit(read_feather_with_head, file_path) for file_path in file_paths]
        chunk_dfs = [future.result() for future in as_completed(futures)]
        
        X = pd.concat(chunk_dfs, axis=1)
        X.columns = [sanitize_feature_name(name) for name in X.columns]
        
        X_train, X_val, y_train, y_val = train_test_split(
            X, target, test_size=0.2, random_state=SEED
        )

        dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=list(set(X_train.columns) & set(ALL_CAT)))
        dval = lgb.Dataset(X_val, label=y_val, categorical_feature=list(set(X_val.columns) & set(ALL_CAT)))

        model = lgb.train(param, dtrain, valid_sets=[dtrain, dval], valid_names=["train", "val"], num_boost_round=1000, callbacks=[lgb.log_evaluation(100), lgb.early_stopping(100)])
        importance_gain = model.feature_importance(importance_type='gain')
        importance_split = model.feature_importance(importance_type='split')

        feature_names = X.columns
        
        y_pred_train = model.predict(X_train)
        y_pred_val = model.predict(X_val)
        auc_train = roc_auc_score(y_train, y_pred_train)
        auc_val = roc_auc_score(y_val, y_pred_val)

        csv_rows = []
        for i, feature_name in enumerate(feature_names):
            csv_rows.append([chunk, feature_name, importance_gain[i], importance_split[i], auc_train, auc_val])
        
        writer.writerows(csv_rows)
        chunk+=1
        
        n += len(file_paths)
        print(n, " / ", feature_paths_size)

Training until validation scores don't improve for 100 rounds
[100]	train's auc: 0.76865	val's auc: 0.748311
[200]	train's auc: 0.782841	val's auc: 0.755175
[300]	train's auc: 0.794925	val's auc: 0.760352
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[400]	train's auc: 0.804654	val's auc: 0.763779
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

In [65]:
n_thread=12 # cpu 6 cores 12 threads

feature_paths = [ROOT + "/data/feature/train/" + sanitize_feature_name(feature).strip() + ".f" for feature in _filtered_low_corr["feature"].tolist()] # thử importance của _filtered_low_corr gain thấp


feature_paths_size = len(feature_paths)
chunk_size = n_thread * 50 # 40-50 là vừa đủ load 70 - 80% RAM, 80% CPU và GPU nhận được lượng data phù hợp tránh nghẽn cổ chai
chunks = [feature_paths[i:i + chunk_size] for i in range(0, len(feature_paths), chunk_size)]

target = pd.read_feather(utils.get_TARGET_path()).head(HEAD)
target.columns = ["TARGET"]

# multi thread
with ThreadPoolExecutor(max_workers=n_thread) as executor, \
    open(os.path.join(ROOT, f".log/_used/7_lightgbm_filtered_low_imp_low_corr.csv"), "w", newline='') as f_selected:
    
    chunk=1
    writer = csv.writer(f_selected)
    writer.writerow(['chunk', 'feature', 'importance_gain', 'importance_split', 'auc_train', 'auc_valid'])
    n = 0
    
    for file_paths in chunks:
        futures = [executor.submit(read_feather_with_head, file_path) for file_path in file_paths]
        chunk_dfs = [future.result() for future in as_completed(futures)]
        
        X = pd.concat(chunk_dfs, axis=1)
        X.columns = [sanitize_feature_name(name) for name in X.columns]
        
        X_train, X_val, y_train, y_val = train_test_split(
            X, target, test_size=0.2, random_state=SEED
        )

        dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=list(set(X_train.columns) & set(ALL_CAT)))
        dval = lgb.Dataset(X_val, label=y_val, categorical_feature=list(set(X_val.columns) & set(ALL_CAT)))

        model = lgb.train(param, dtrain, valid_sets=[dtrain, dval], valid_names=["train", "val"], num_boost_round=1000, callbacks=[lgb.log_evaluation(1000), lgb.early_stopping(100)])
        importance_gain = model.feature_importance(importance_type='gain')
        importance_split = model.feature_importance(importance_type='split')

        feature_names = X.columns
        
        y_pred_train = model.predict(X_train)
        y_pred_val = model.predict(X_val)
        auc_train = roc_auc_score(y_train, y_pred_train)
        auc_val = roc_auc_score(y_val, y_pred_val)

        csv_rows = []
        for i, feature_name in enumerate(feature_names):
            csv_rows.append([chunk, feature_name, importance_gain[i], importance_split[i], auc_train, auc_val])
        
        writer.writerows(csv_rows)
        chunk+=1
        
        n += len(file_paths)
        print(n, " / ", feature_paths_size)

Training until validation scores don't improve for 100 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

In [68]:
n_thread=12 # cpu 6 cores 12 threads

feature_paths = [ROOT + "/data/feature/train/" + sanitize_feature_name(feature).strip() + ".f" for feature in filtered_low_corr["feature"].tolist()] # thử importance của filtered_low_corr gain cao


feature_paths_size = len(feature_paths)
chunk_size = n_thread * 50 # 40-50 là vừa đủ load 70 - 80% RAM, 80% CPU và GPU nhận được lượng data phù hợp tránh nghẽn cổ chai
chunks = [feature_paths[i:i + chunk_size] for i in range(0, len(feature_paths), chunk_size)]

target = pd.read_feather(utils.get_TARGET_path()).head(HEAD)
target.columns = ["TARGET"]

# multi thread
with ThreadPoolExecutor(max_workers=n_thread) as executor, \
    open(os.path.join(ROOT, f".log/_used/7_lightgbm_filtered_high_imp_low_corr.csv"), "w", newline='') as f_selected:
    
    chunk=1
    writer = csv.writer(f_selected)
    writer.writerow(['chunk', 'feature', 'importance_gain', 'importance_split', 'auc_train', 'auc_valid'])
    n = 0
    
    for file_paths in chunks:
        futures = [executor.submit(read_feather_with_head, file_path) for file_path in file_paths]
        chunk_dfs = [future.result() for future in as_completed(futures)]
        
        X = pd.concat(chunk_dfs, axis=1)
        X.columns = [sanitize_feature_name(name) for name in X.columns]
        
        X_train, X_val, y_train, y_val = train_test_split(
            X, target, test_size=0.2, random_state=SEED
        )

        dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=list(set(X_train.columns) & set(ALL_CAT)))
        dval = lgb.Dataset(X_val, label=y_val, categorical_feature=list(set(X_val.columns) & set(ALL_CAT)))

        model = lgb.train(param, dtrain, valid_sets=[dtrain, dval], valid_names=["train", "val"], num_boost_round=1000, callbacks=[lgb.log_evaluation(100), lgb.early_stopping(100)])
        importance_gain = model.feature_importance(importance_type='gain')
        importance_split = model.feature_importance(importance_type='split')

        feature_names = X.columns
        
        y_pred_train = model.predict(X_train)
        y_pred_val = model.predict(X_val)
        auc_train = roc_auc_score(y_train, y_pred_train)
        auc_val = roc_auc_score(y_val, y_pred_val)

        csv_rows = []
        for i, feature_name in enumerate(feature_names):
            csv_rows.append([chunk, feature_name, importance_gain[i], importance_split[i], auc_train, auc_val])
        
        writer.writerows(csv_rows)
        chunk+=1
        
        n += len(file_paths)
        print(n, " / ", feature_paths_size)

Training until validation scores don't improve for 100 rounds
[100]	train's auc: 0.768958	val's auc: 0.748702
[200]	train's auc: 0.783133	val's auc: 0.755568
[300]	train's auc: 0.795169	val's auc: 0.76047
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[400]	train's auc: 0.804707	val's auc: 0.763592
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i